# Agent: Control Agent (Decision Logic)

**Purpose:** Monitor trigger events and water level readings; decide on evacuation alerts.

**Decision Logic:**
- If water_level ≥ 1.0m → Issue HIGH alert (evacuation)
- If water_level < 1.0m → Issue LOW alert (all-clear)

**Subscribes to:** `city/flood/trigger` topic

**Publishes to:** `city/flood/control/command` topic as JSON `ControlCommand` messages

In [45]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import TriggerEvent, ControlCommand

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt

print(f"MQTT Broker: {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Base Topic: {mqtt_cfg.base_topic}")

# Control parameters
WATER_THRESHOLD = 1.0     # Alert threshold in meters
BASELINE_WATER = 0.2      # Normal water level (m)
RISE_RATE = 0.20          # Water rise per second (m/s)
RISE_DURATION_S = 30.0    # Rising phase duration (s)
FALL_DURATION_S = 15.0    # Falling phase duration (s)
CYCLE_DURATION_S = RISE_DURATION_S + FALL_DURATION_S
FLOOD_WATER = BASELINE_WATER + (RISE_RATE * RISE_DURATION_S)  # 6.2m max
CONTROL_INTERVAL_S = 1.0

# State tracking
current_trigger = None
current_water_level = BASELINE_WATER
current_alert_status = False
update_count = 0
cycle_elapsed_s = 0.0
current_phase = "rising"

print(f"\nControl agent configured:")
print(f"  Water threshold: {WATER_THRESHOLD}m")
print(f"  Baseline level: {BASELINE_WATER}m")
print(f"  Flood level: {FLOOD_WATER}m")
print(f"  Cycle: rise {RISE_DURATION_S:.0f}s at {RISE_RATE:.2f}m/s, fall {FALL_DURATION_S:.0f}s")

MQTT Broker: 127.0.0.1:1883
Base Topic: simulated-city

Control agent configured:
  Water threshold: 1.0m
  Baseline level: 0.2m
  Flood level: 6.2m
  Cycle: rise 30s at 0.20m/s, fall 15s


In [46]:
# Connect to MQTT broker
connector = MqttConnector(mqtt_cfg, client_id_suffix='control')
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    publisher = MqttPublisher(connector)
else:
    print("✗ Failed to connect to MQTT broker")
    publisher = None


def on_trigger_message(client, userdata, msg):
    """Callback when trigger event is received."""
    global current_trigger
    try:
        payload = msg.payload.decode()
        data = json.loads(payload)
        current_trigger = TriggerEvent.from_json(data)
        print(f"  → Trigger input: {current_trigger.severity} {current_trigger.event}")
    except Exception as e:
        print(f"Error parsing trigger: {e}")


# Subscribe to trigger events
if publisher:
    trigger_topic = make_topic(mqtt_cfg, "flood", "trigger")
    connector.client.subscribe(trigger_topic, qos=1)
    connector.client.message_callback_add(trigger_topic, on_trigger_message)
    print(f"Subscribed to: {trigger_topic}")

✓ Connected to MQTT broker
Subscribed to: simulated-city/flood/trigger


In [47]:
def publish_control_command(severity: str):
    """Publish a control command to Response agent."""
    if not publisher:
        return False
    
    cmd = ControlCommand(
        action="alert",
        target="all",
        parameters={"severity": severity, "threshold": WATER_THRESHOLD},
        timestamp=datetime.now(timezone.utc).isoformat()
    )
    
    topic = make_topic(mqtt_cfg, "flood", "control", "command")
    payload = json.dumps(cmd.to_json())
    
    try:
        publisher.publish_json(topic, payload, qos=1)
        return True
    except Exception as e:
        print(f"Error publishing control command: {e}")
        return False


def update_water_level():
    """Simulate repeating flood cycle: rise 30s, fall 15s."""
    global current_water_level, cycle_elapsed_s, current_phase

    cycle_position = cycle_elapsed_s % CYCLE_DURATION_S

    if cycle_position < RISE_DURATION_S:
        current_phase = "rising"
        current_water_level = BASELINE_WATER + (RISE_RATE * cycle_position)
    else:
        current_phase = "falling"
        fall_time = cycle_position - RISE_DURATION_S
        fall_rate = (FLOOD_WATER - BASELINE_WATER) / FALL_DURATION_S
        current_water_level = max(FLOOD_WATER - (fall_rate * fall_time), BASELINE_WATER)

    cycle_elapsed_s = (cycle_elapsed_s + CONTROL_INTERVAL_S) % CYCLE_DURATION_S
    return current_water_level


def check_alert_status() -> tuple[bool, str]:
    """Determine if alert should be active based on water level."""
    if current_water_level >= WATER_THRESHOLD:
        return True, "high"
    else:
        return False, "low"


def update_control_state():
    """Update control state and issue commands if needed."""
    global current_alert_status, update_count
    
    water_level = update_water_level()
    new_alert_status, severity = check_alert_status()
    
    update_count += 1
    
    if new_alert_status != current_alert_status:
        if new_alert_status:
            print(f"  ⚠️  ALERT: {water_level:.2f}m >= {WATER_THRESHOLD}m (HIGH severity)")
            publish_control_command("high")
        else:
            print(f"  ✓ CLEAR: {water_level:.2f}m < {WATER_THRESHOLD}m (LOW severity)")
            publish_control_command("low")
        current_alert_status = new_alert_status

In [ ]:
print("\n⚙️ Starting control loop...")
print(f"Water threshold: {WATER_THRESHOLD}m")
print(f"Update interval: {CONTROL_INTERVAL_S}s")
print("Press Ctrl+C to stop\n")

try:
    while True:
        update_control_state()
        timestamp = datetime.now(timezone.utc).strftime('%H:%M:%S')
        print(f"[{timestamp}] Phase={current_phase:7s} | Water={current_water_level:.2f}m | Alert={current_alert_status}")
        time.sleep(CONTROL_INTERVAL_S)
        
except KeyboardInterrupt:
    print("\n\n✓ Control agent stopped")
finally:
    if connector:
        connector.disconnect()


⚙️ Starting control loop...
Water threshold: 1.0m
Update interval: 1.0s
Press Ctrl+C to stop

[13:26:05] Phase=rising  | Water=0.20m | Alert=False
[13:26:06] Phase=rising  | Water=0.40m | Alert=False
[13:26:07] Phase=rising  | Water=0.60m | Alert=False
[13:26:08] Phase=rising  | Water=0.80m | Alert=False
  ⚠️  ALERT: 1.00m >= 1.0m (HIGH severity)
[13:26:09] Phase=rising  | Water=1.00m | Alert=True
[13:26:10] Phase=rising  | Water=1.20m | Alert=True
[13:26:11] Phase=rising  | Water=1.40m | Alert=True
[13:26:12] Phase=rising  | Water=1.60m | Alert=True
[13:26:13] Phase=rising  | Water=1.80m | Alert=True
[13:26:14] Phase=rising  | Water=2.00m | Alert=True
[13:26:15] Phase=rising  | Water=2.20m | Alert=True
[13:26:16] Phase=rising  | Water=2.40m | Alert=True
[13:26:17] Phase=rising  | Water=2.60m | Alert=True
[13:26:18] Phase=rising  | Water=2.80m | Alert=True
[13:26:19] Phase=rising  | Water=3.00m | Alert=True
[13:26:20] Phase=rising  | Water=3.20m | Alert=True
[13:26:21] Phase=rising  |

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[15:31:38] Phase=rising  | Water=4.80m | Alert=True
[15:31:39] Phase=rising  | Water=5.00m | Alert=True


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[17:30:41] Phase=rising  | Water=5.20m | Alert=True
[17:30:42] Phase=rising  | Water=5.40m | Alert=True
[17:30:43] Phase=rising  | Water=5.60m | Alert=True
[17:30:44] Phase=rising  | Water=5.80m | Alert=True
[17:30:45] Phase=rising  | Water=6.00m | Alert=True
[17:30:46] Phase=falling | Water=6.20m | Alert=True
[17:30:47] Phase=falling | Water=5.80m | Alert=True
[17:30:48] Phase=falling | Water=5.40m | Alert=True
[17:30:49] Phase=falling | Water=5.00m | Alert=True
[17:30:50] Phase=falling | Water=4.60m | Alert=True
[17:30:51] Phase=falling | Water=4.20m | Alert=True
[17:30:52] Phase=falling | Water=3.80m | Alert=True
[17:30:53] Phase=falling | Water=3.40m | Alert=True
[17:30:54] Phase=falling | Water=3.00m | Alert=True
[17:30:55] Phase=falling | Water=2.60m | Alert=True
[17:30:56] Phase=falling | Water=2.20m | Alert=True
[17:30:57] Phase=falling | Water=1.80m | Alert=True
[17:30:58] Phase=falling | Water=1.40m | Alert=True
[17:30:59] Phase=falling | Water=1.00m | Alert=True
  ✓ CLEAR: 0

...

[20:05:18] Phase=falling | Water=1.80m | Alert=True
[20:05:19] Phase=falling | Water=1.40m | Alert=True
[20:05:20] Phase=falling | Water=1.00m | Alert=True
  ✓ CLEAR: 0.60m < 1.0m (LOW severity)
[20:05:21] Phase=falling | Water=0.60m | Alert=False
[20:05:22] Phase=rising  | Water=0.20m | Alert=False
[20:05:23] Phase=rising  | Water=0.40m | Alert=False
[20:05:24] Phase=rising  | Water=0.60m | Alert=False
[20:05:26] Phase=rising  | Water=0.80m | Alert=False
  ⚠️  ALERT: 1.00m >= 1.0m (HIGH severity)
[20:05:27] Phase=rising  | Water=1.00m | Alert=True
[20:05:28] Phase=rising  | Water=1.20m | Alert=True
[20:05:29] Phase=rising  | Water=1.40m | Alert=True
[20:05:30] Phase=rising  | Water=1.60m | Alert=True
[20:05:31] Phase=rising  | Water=1.80m | Alert=True
[20:05:32] Phase=rising  | Water=2.00m | Alert=True
[20:05:33] Phase=rising  | Water=2.20m | Alert=True
[20:05:35] Phase=rising  | Water=2.40m | Alert=True
[20:05:36] Phase=rising  | Water=2.60m | Alert=True
[20:05:37] Phase=rising  | Wa

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[21:49:57] Phase=rising  | Water=4.80m | Alert=True


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[22:42:26] Phase=rising  | Water=5.00m | Alert=True
[22:42:27] Phase=rising  | Water=5.20m | Alert=True
[22:42:28] Phase=rising  | Water=5.40m | Alert=True
[22:42:29] Phase=rising  | Water=5.60m | Alert=True
[22:42:30] Phase=rising  | Water=5.80m | Alert=True
[22:42:31] Phase=rising  | Water=6.00m | Alert=True
[22:42:32] Phase=falling | Water=6.20m | Alert=True
[22:42:33] Phase=falling | Water=5.80m | Alert=True
[22:42:34] Phase=falling | Water=5.40m | Alert=True
[22:42:36] Phase=falling | Water=5.00m | Alert=True
[22:42:37] Phase=falling | Water=4.60m | Alert=True
[22:42:38] Phase=falling | Water=4.20m | Alert=True
[22:42:39] Phase=falling | Water=3.80m | Alert=True
[22:42:40] Phase=falling | Water=3.40m | Alert=True
[22:42:41] Phase=falling | Water=3.00m | Alert=True
[22:42:42] Phase=falling | Water=2.60m | Alert=True
[22:42:43] Phase=falling | Water=2.20m | Alert=True
[22:42:44] Phase=falling | Water=1.80m | Alert=True
[22:42:45] Phase=falling | Water=1.40m | Alert=True
[22:42:46] P

In [ ]:
UPDATE_INTERVAL = 1.0  # Update decision every 1 second
FALL_RATE = (FLOOD_WATER_LEVEL - BASELINE_WATER_LEVEL) / FALL_DURATION_S  # meters per second

print("Starting control loop...")
print(f"Update interval: {UPDATE_INTERVAL}s\n")

loop_count = 0
last_update_time = time.time()

try:
    while True:
        loop_count += 1
        current_time = time.time()
        elapsed = current_time - last_update_time

        with state["lock"]:
            cycle_elapsed = state["cycle_elapsed_s"]
            was_alerted = state["alerted"]

        cycle_position = cycle_elapsed % CYCLE_DURATION_S
        if cycle_position < RISE_DURATION_S:
            phase = "rising"
            desired_level = BASELINE_WATER_LEVEL + (RISE_RATE * cycle_position)
        else:
            phase = "falling"
            fall_time = cycle_position - RISE_DURATION_S
            desired_level = max(FLOOD_WATER_LEVEL - (FALL_RATE * fall_time), BASELINE_WATER_LEVEL)

        should_alert = desired_level >= WATER_LEVEL_THRESHOLD

        with state["lock"]:
            state["current_water_level"] = desired_level
            state["cycle_elapsed_s"] = (cycle_elapsed + elapsed) % CYCLE_DURATION_S

        if should_alert and not was_alerted:
            print(f"[{loop_count}] ⚠️  ALERT: Water level {desired_level:.2f}m >= threshold {WATER_LEVEL_THRESHOLD}m")
            publish_alert_command("alert", "high")
            with state["lock"]:
                state["alerted"] = True
        elif not should_alert and was_alerted:
            print(f"[{loop_count}] ✓ ALL-CLEAR: Water level {desired_level:.2f}m < threshold")
            publish_alert_command("alert", "low")
            with state["lock"]:
                state["alerted"] = False
        else:
            with state["lock"]:
                alert_state = state["alerted"]
            print(f"[{loop_count}] Water: {desired_level:.2f}m | Phase: {phase} | Alert: {alert_state}")

        last_update_time = current_time
        time.sleep(UPDATE_INTERVAL)

except KeyboardInterrupt:
    print("\n\nControl loop stopped by user.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

Starting control loop...
Update interval: 1.0s

[1] Water: 0.20m | Trigger: None | Alert: False
[2] Water: 0.20m | Trigger: None | Alert: False
[3] Water: 0.20m | Trigger: None | Alert: False
[4] Water: 0.20m | Trigger: None | Alert: False
[5] Water: 0.20m | Trigger: None | Alert: False
[6] Water: 0.20m | Trigger: None | Alert: False
[7] Water: 0.20m | Trigger: None | Alert: False
[8] Water: 0.20m | Trigger: None | Alert: False
[9] Water: 0.20m | Trigger: None | Alert: False
[10] Water: 0.20m | Trigger: None | Alert: False
[11] Water: 0.20m | Trigger: None | Alert: False
[12] Water: 0.20m | Trigger: None | Alert: False
[13] Water: 0.20m | Trigger: None | Alert: False
[14] Water: 0.20m | Trigger: None | Alert: False
[15] Water: 0.20m | Trigger: None | Alert: False
[16] Water: 0.20m | Trigger: None | Alert: False
[17] Water: 0.20m | Trigger: None | Alert: False
[18] Water: 0.20m | Trigger: None | Alert: False
[19] Water: 0.20m | Trigger: None | Alert: False
[20] Water: 0.20m | Trigger: N

## Control Loop

Main decision loop that:
1. Checks trigger severity
2. Updates water level (ramp up/down)
3. Issues alerts/all-clear commands based on threshold

In [ ]:
def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def publish_alert_command(action, severity):
    """Publish a control command to response agents."""
    cmd = ControlCommand(
        action=action,
        target="all_pedestrians",
        parameters={"severity": severity},
        timestamp=get_iso_timestamp()
    )
    topic = make_topic(mqtt_cfg, "control", "command")
    payload = json.dumps(cmd.to_json())
    publisher.publish_json(topic, payload, qos=1)
    print(f"  → Published control command: {action} ({severity})")
    return cmd

def on_trigger_message(client, userdata, msg):
    """Callback when trigger event received."""
    try:
        data = json.loads(msg.payload.decode())
        evt = TriggerEvent.from_json(data)
        
        with state["lock"]:
            state["last_trigger_severity"] = evt.severity
        
        print(f"[{evt.timestamp}] Trigger received: {evt.severity.upper()} {evt.event}")
    except Exception as e:
        print(f"Error parsing trigger: {e}")

# Register MQTT callbacks
connector.client.on_message = on_trigger_message
trigger_topic = make_topic(mqtt_cfg, "trigger")
connector.client.subscribe(trigger_topic)

print(f"Subscribed to: {trigger_topic}")

Subscribed to: simulated-city/trigger


## Decision Logic and Callbacks

This section defines the logic for:
1. Receiving trigger events
2. Updating water level simulation
3. Issuing alert commands to response agents

In [ ]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="control")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

✓ Connected to MQTT broker


## Connect to MQTT Broker

In [ ]:
import time
import json
import threading
from datetime import datetime, timezone

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import TriggerEvent, ControlCommand

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

# Flood-cycle configuration
WATER_LEVEL_THRESHOLD = 1.0  # meters - trigger evacuation
BASELINE_WATER_LEVEL = 0.2   # meters baseline
RISE_RATE = 0.20             # meters per second
RISE_DURATION_S = 30.0       # seconds
FALL_DURATION_S = 15.0       # seconds
CYCLE_DURATION_S = RISE_DURATION_S + FALL_DURATION_S
FLOOD_WATER_LEVEL = BASELINE_WATER_LEVEL + (RISE_RATE * RISE_DURATION_S)  # 6.2m

# State tracking
state = {
    "last_trigger_severity": None,
    "current_water_level": BASELINE_WATER_LEVEL,
    "alerted": False,
    "cycle_elapsed_s": 0.0,
    "lock": threading.Lock()
}

print("Control agent configured:")
print(f"  Water threshold: {WATER_LEVEL_THRESHOLD}m")
print(f"  Baseline level: {BASELINE_WATER_LEVEL}m")
print(f"  Flood level: {FLOOD_WATER_LEVEL}m")
print(f"  Cycle: rise {RISE_DURATION_S:.0f}s at {RISE_RATE:.2f}m/s, fall {FALL_DURATION_S:.0f}s")

Control agent configured:
  Water threshold: 1.0m
  Baseline level: 0.2m
  Flood level: 6.2m
  Cycle: rise 30s at 0.20m/s, fall 15s


## Setup and Configuration

# Agent: Control (Decision Logic)

**Role:** Central decision engine for the flood response system.

**Responsibilities:**
1. Subscribe to trigger events and sensor data
2. Simulate cyclic water level changes
3. Monitor water level threshold (1m)
4. Issue evacuation commands when threshold exceeded
5. Coordinate with response agents

**Decision Logic:**
- Water rises from 0.2m at 0.20m/s for 30 seconds
- Water then falls back to 0.2m over 15 seconds
- Water >= 1m → Issue "alert" command (evacuation)
- Water < 1m → Issue "all-clear" command
- Cycle repeats continuously

# Agent Control
Logic notebook that subscribes to observer data and trigger events, then emits `ControlCommand` messages.